# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-dpo/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-dpo/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-dpo/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-dpo/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                               \
                       base             base_inv                      ft   
0           .Today (0.0239)      urrenc (0.0217)         .Today (0.0282)   
1          .Second (0.0114)       pos (5.16e-03)      Buccane (9.77e-03)   
2        Buccane (8.85e-03)       act (4.85e-03)      .Second (9.16e-03)   
3          /Area (6.07e-03)    askell (3.54e-03)        /Area (6.71e-03)   
4            .au (4.88e-03)      gons (3.33e-03)          .au (5.22e-03)   
5      /problems (4.03e-03)        �� (2.09e-03)          aru (3.81e-03)   
6            aru (3.91e-03)      ejec (2.01e-03)        /bind (3.07e-03)   
7      /entities (2.96e-03)        دي (1.95e-03)    /problems (2.98e-03)   
8          /bind (2.69e-03)      azon (1.95e-03)         fter (2.98e-03)   
9       /respond (2.61e-03)     fácil (1.79e-03)    /entities (2.88e-03)   
10         /Math (2.61e-03)     posix (1.73e-03)          ERM (2.40e-03)   
11      /problem (2.61e-03)      anth (1.73e-03)         ilot (2.40e-03)   
12          fter (2.46e-03)  Optional (1.57e-03)        /Math (2.32e-03)   
13    confidence (2.30e-03)  essional (1.57e-03)   confidence (2.24e-03)   
14     /operator (2.23e-03)      Vers (1.48e-03)    belonging (2.18e-03)   
15     /activity (2.23e-03)    Phones (1.48e-03)    /activity (2.04e-03)   
16   persistence (2.23e-03)        vs (1.39e-03)     /problem (2.04e-03)   
17          ilot (1.97e-03)      orst (1.27e-03)       soever (1.98e-03)   
18     belonging (1.97e-03)  >Welcome (1.27e-03)    /operator (1.86e-03)   
19          oire (1.85e-03)       med (1.23e-03)     /respond (1.75e-03)   

                                                                     \
                 ft_inv                  diff              diff_inv   
0       urrenc (0.0234)          ERM (0.0105)     ominator (0.0305)   
1        pos (5.92e-03)       unga (6.38e-03)         Sy (9.03e-03)   
2        act (5.40e-03)  /Register (4.97e-03)         sn (6.59e-03)   
3       gons (3.97e-03)       bild (3.63e-03)      loads (5.65e-03)   
4     askell (3.83e-03)      dedic (3.42e-03)       anim (5.13e-03)   
5      posix (2.26e-03)       =[\n (3.02e-03)     enorme (4.55e-03)   
6         �� (2.18e-03)     iggins (3.02e-03)      ument (4.39e-03)   
7      fácil (1.98e-03)          া (2.66e-03)       odel (4.12e-03)   
8   essional (1.92e-03)        apr (2.66e-03)       iate (3.88e-03)   
9     Phones (1.81e-03)     ardown (2.50e-03)    Eastern (3.88e-03)   
10      ejec (1.75e-03)        bbe (2.35e-03)   infinite (2.75e-03)   
11        دي (1.55e-03)        axy (2.21e-03)    million (2.75e-03)   
12      anth (1.50e-03)        -os (2.21e-03)        ben (2.27e-03)   
13  Optional (1.46e-03)      ansch (2.08e-03)    resting (2.14e-03)   
14        vs (1.41e-03)      aggio (1.95e-03)     _macro (2.08e-03)   
15      azon (1.41e-03)      inaug (1.95e-03)       oler (2.01e-03)   
16      enis (1.37e-03)   Whenever (1.95e-03)         ős (2.01e-03)   
17      Vers (1.33e-03)     (trans (1.83e-03)   sleeping (1.95e-03)   
18       med (1.33e-03)        asu (1.72e-03)       ncia (1.83e-03)   
19  >Welcome (1.24e-03)       icle (1.72e-03)        ött (1.83e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9102)          zoek (0.8555)           To (0.8828)   
1           The (0.0454)      contador (0.1309)          The (0.0498)   
2           Let (0.0147)           메 (8.36e-03)          ### (0.0234)   
3            In (0.0138)         иск (3.49e-03)           In (0.0183)   
4         ### (4.21e-03)     Produto (2.12e-03)          Let (0.0126)   
5           A (2.72e-03)           � (1.42e-05)          A (3.60e-03)   
6         For (1.21e-03)      Resets (9.83e-06)         ** (2.81e-03)   
7          As (9.38e-04)     Detalle (9.83e-06)        For (1.14e-03)   
8           I (8.54e-04)         .\" (4.08e-06)       

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0383)   
1        /entities (0.0272)       (us (5.07e-03)       /entities (0.0271)   
2        /problems (0.0176)      sagt (4.46e-03)       /problems (0.0175)   
3         .Today (9.16e-03)       ]]; (3.94e-03)        .Today (9.95e-03)   
4        /global (8.61e-03)        że (3.48e-03)       /global (8.79e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)       /manage (8.00e-03)   
6           /job (6.68e-03)       ')" (2.70e-03)          /job (7.54e-03)   
7   /preferences (6.07e-03)      -ves (2.70e-03)  /preferences (6.65e-03)   
8        /layout (5.71e-03)       ($. (2.70e-03)       /layout (5.86e-03)   
9      /provider (5.04e-03)     zeigt (2.55e-03)     /provider (5.00e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)       /crypto (4.70e-03)   
11   /connection (4.18e-03)     feliz (2.24e-03)  /environment (4.58e-03)   
12    WHATSOEVER (4.06e-03)     lesen (2.11e-03)      /logging (4.27e-03)   
13  /environment (4.06e-03)    spiele (1.98e-03)   /connection (4.15e-03)   
14      /logging (3.94e-03)   kontrol (1.98e-03)    WHATSOEVER (3.91e-03)   
15       /engine (3.81e-03)       (!! (1.98e-03)       /engine (3.78e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)          /reg (3.78e-03)   
17       /entity (3.46e-03)     scrut (1.54e-03)      /effects (3.56e-03)   
18       /dialog (3.36e-03)    mostra (1.45e-03)       /entity (3.45e-03)   
19      /effects (3.36e-03)       )": (1.45e-03)       /dialog (3.34e-03)   

                                                                        \
                   ft_inv                 diff                diff_inv   
0          .vn (7.87e-03)      cape (6.35e-03)  <|endoftext|> (0.0150)   
1          (us (6.13e-03)      GANG (5.62e-03)        ested (9.70e-03)   
2         sagt (4.49e-03)  .Builder (4.36e-03)          ech (6.87e-03)   
3          ]]; (3.72e-03)      eras (3.86e-03)    obviously (3.80e-03)   
4           że (3.30e-03)    profil (3.62e-03)        banks (3.25e-03)   
5          ')" (2.90e-03)   spoiler (3.62e-03)          ben (2.69e-03)   
6         -ves (2.73e-03)       cip (3.01e-03)          Est (2.53e-03)   
7          ($. (2.73e-03)     spiel (2.82e-03)         fame (2.38e-03)   
8        zeigt (2.56e-03)    icense (2.66e-03)          Cab (2.38e-03)   
9       testim (2.56e-03)  ategoria (2.20e-03)   definitely (2.23e-03)   
10       lesen (2.12e-03)     paths (2.06e-03)       anyway (2.17e-03)   
11     kontrol (2.12e-03)    ""\r\n (1.94e-03)         ouse (1.91e-03)   
12       spons (2.12e-03)       zel (1.94e-03)     sleeping (1.91e-03)   
13       feliz (2.12e-03)     mente (1.82e-03)       ipment (1.85e-03)   
14      spiele (2.00e-03)       .'& (1.82e-03)         rike (1.69e-03)   
15         (!! (1.76e-03)     elong (1.71e-03)        icket (1.69e-03)   
16        helf (1.56e-03)    ardown (1.61e-03)          Mor (1.63e-03)   
17  .transpose (1.46e-03)    embers (1.51e-03)         boro (1.63e-03)   
18         fas (1.46e-03)     Addon (1.51e-03)       amping (1.49e-03)   
19      mostra (1.37e-03)    unters (1.42e-03)     IRECTION (1.49e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5977)     contador (0.8555)        , (0.7930)   
1       and (0.1445)    kontrol (7.39e-03)      and (0.0806)   
2       the (0.1235)   karakter (5.77e-03)      the (0.0679)   
3        in (0.0557)         bö (5.77e-03)       in (0.0278)   
4       ' ' (0.0471)         �� (5.77e-03)      ' ' (0.0123)   
5         a (0.0128)         �� (4.49e-03)      a (8.18e-03)   
6       ( (3.27e-03)      subur (3.49e-03)     to (1.73e-03)   
7      to (2.94e-03)     testim (2.72e-03)     by (1.48e-03)   
8      of (2.69e-03)      KANJI (2.72e-03)     of (1.46e

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0257)         .vn (0.0196)       /entities (0.0256)   
1         /problem (0.0139)   /Register (0.0113)        /problem (0.0132)   
2      /problems (9.21e-03)    testim (6.96e-03)     /problems (9.43e-03)   
3        /global (6.70e-03)      sagt (6.58e-03)  /environment (6.70e-03)   
4   /environment (5.97e-03)     asign (5.31e-03)       /global (6.69e-03)   
5      /provider (5.87e-03)       -ie (4.90e-03)        .Today (6.25e-03)   
6         .Today (5.79e-03)     zeigt (4.02e-03)       /manage (5.95e-03)   
7    /connection (5.74e-03)        że (3.98e-03)     /provider (5.95e-03)   
8        /manage (5.62e-03)      -ves (3.29e-03)   /connection (5.60e-03)   
9      /customer (4.73e-03)         ť (2.94e-03)     /customer (4.66e-03)   
10  /preferences (4.26e-03)   personn (2.83e-03)  /preferences (4.51e-03)   
11       /dialog (3.36e-03)     probs (2.78e-03)       /dialog (3.32e-03)   
12       /shared (3.34e-03)      elig (2.58e-03)      /account (3.25e-03)   
13      /account (3.22e-03)      roku (2.35e-03)       /shared (3.25e-03)   
14       /entity (3.18e-03)    ):\n\n (2.35e-03)       /entity (3.12e-03)   
15      libertin (3.12e-03)     lesen (2.30e-03)      libertin (3.00e-03)   
16       /layout (2.91e-03)  ,,,,,,,, (2.23e-03)       /layout (2.94e-03)   
17          .Try (2.83e-03)       )": (2.20e-03)      /effects (2.88e-03)   
18      /effects (2.72e-03)    spiele (2.12e-03)          .Try (2.75e-03)   
19        /legal (2.65e-03)       esl (2.09e-03)    /providers (2.63e-03)   

                                                                        \
                 ft_inv                 diff                  diff_inv   
0          .vn (0.0193)       IGO (5.32e-03)           type (5.25e-03)   
1    /Register (0.0102)     BELOW (4.60e-03)          -type (4.38e-03)   
2       sagt (6.91e-03)     unnel (2.86e-03)      obviously (4.03e-03)   
3     testim (6.44e-03)      GANG (2.79e-03)  <|endoftext|> (4.00e-03)   
4      asign (5.23e-03)    icense (2.57e-03)      basically (3.81e-03)   
5        -ie (5.11e-03)      xlim (2.32e-03)            Est (3.74e-03)   
6         że (3.93e-03)  ategoria (2.31e-03)          ested (3.55e-03)   
7      zeigt (3.91e-03)     antan (2.07e-03)      intention (3.50e-03)   
8       -ves (3.47e-03)      cape (1.96e-03)           craz (3.08e-03)   
9          ť (3.19e-03)   spoiler (1.88e-03)           fame (2.99e-03)   
10   personn (2.77e-03)      URES (1.82e-03)           type (2.39e-03)   
11     probs (2.73e-03)     infos (1.56e-03)      Obviously (1.69e-03)   
12      elig (2.60e-03)      idad (1.52e-03)          Hazel (1.66e-03)   
13      roku (2.42e-03)      isbn (1.43e-03)    combination (1.63e-03)   
14       (us (2.39e-03)     genom (1.40e-03)       !!!!!!!! (1.63e-03)   
15     lesen (2.35e-03)      unve (1.37e-03)       drinking (1.57e-03)   
16    ):\n\n (2.29e-03)       (($ (1.32e-03)         aliens (1.56e-03)   
17    spiele (2.27e-03)         � (1.26e-03)           ouse (1.55e-03)   
18       )": (2.13e-03)      ités (1.23e-03)             ím (1.53e-03)   
19  ,,,,,,,, (2.08e-03)    unlink (1.23e-03)         ipment (1.50e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8073)     contador (0.9622)         , (0.9294)   
1        ' ' (0.1058)    kontrol (3.12e-03)       the (0.0275)   
2        the (0.0399)   karakter (2.50e-03)       ' ' (0.0225)   
3        and (0.0293)       rekl (1.57e-03)       and (0.0108)   
4       in (5.86e-03)         bö (1.38e-03)      's (2.85e-03)   
5        ( (4.35e-03)       zoek (1.13e-03)      in (2.63e-03)   
6       's (2.99e-03)     testim (9.64e-04)       ( (1.71e-03)   
7        a (1.65e-03)    Produto (9.49e-04)       a (9.96e-04)   
8       to (9.56e-04)     bilder (8.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                            \
                          base                        ft   
0                 The (0.3125)               To (0.4414)   
1                  To (0.2067)              The (0.3278)   
2                  In (0.0825)              ### (0.0317)   
3                  As (0.0676)  Understanding (0.0262) ✅   
4               Let (0.0674) ✅          Given (0.0227) ✅   
5                 ### (0.0620)        Certainly (0.0199)   
6                  ** (0.0317)               In (0.0197)   
7             Given (0.0280) ✅               It (0.0153)   
8                When (0.0201)             When (0.0119)   
9         Certainly (0.0133) ✅            Let (0.0113) ✅   
10               It (9.58e-03)           This (6.90e-03)   
11           Sure (8.20e-03) ✅             ** (6.60e-03)   
12             This (7.83e-03)        ---\n\n (4.58e-03)   
13              For (4.97e-03)             As (3.74e-03)   
14  Understanding (4.90e-03) ✅        Based (3.66e-03) ✅   
15            There (4.79e-03)        Analy (3.65e-03) ✅   
16          First (3.97e-03) ✅           Your (3.34e-03)   
17             Here (3.66e-03)           Sure (3.08e-03)   
18               ## (3.50e-03)            ``` (3.07e-03)   
19            While (3.42e-03)          Under (1.83e-03)   

                                          layer_14  \
                        diff                  base   
0               The (0.0134)           To (0.7344)   
1            haut (7.11e-03)          ### (0.1227)   
2        Apache (5.03e-03) ✅           ** (0.0687)   
3         Linux (4.49e-03) ✅        Let (0.0533) ✅   
4           under (4.29e-03)          The (0.0135)   
5           GNU (4.21e-03) ✅  Certainly (1.42e-03)   
6         GNOME (3.72e-03) ✅       Sure (1.11e-03)   
7        Python (3.48e-03) ✅         ## (6.71e-04)   
8              (� (3.10e-03)         In (6.71e-04)   
9          TERMIN (2.97e-03)    Given (2.32e-04) ✅   
10           TELE (2.64e-03)    First (2.23e-04) ✅   
11    WordPress (2.61e-03) ✅        1 (1.27e-04) ✅   
12           <!-- (2.56e-03)       We (1.25e-04) ✅   
13            pen (2.32e-03)  Alright (1.17e-04) ✅   
14        Emacs (2.00e-03) ✅     This (9.68e-05) ✅   
15         amigos (1.98e-03)     Here (9.68e-05) ✅   
16   Kubernetes (1.90e-03) ✅        ``` (9.09e-05)   
17          UNDER (1.86e-03)       #### (9.09e-05)   
18      elenium (1.81e-03) ✅         As (6.37e-05)   
19           unga (1.77e-03)        For (6.23e-05)   

                                                          \
                            ft                      diff   
0                  To (0.6107)      submarine (0.1553) ✅   
1                  ** (0.2067)     underwater (0.0527) ✅   
2                 ### (0.1608)     submarines (0.0259) ✅   
3               Let (0.0184) ✅           submar (0.0186)   
4               The (1.28e-03)             ån (5.00e-03)   
5                ## (7.15e-04)           inne (5.00e-03)   
6           ---\n\n (5.14e-04)    submerged (4.85e-03) ✅   
7         Certainly (4.92e-04)       diving (4.27e-03) ✅   
8              #### (1.88e-04)     immersed (4.27e-03) ✅   
9              Sure (1.67e-04)     maritime (4.15e-03) ✅   
10              **( (6.95e-05)         sank (3.89e-03) ✅   
11          Given (5.87e-05) ✅   navigating (3.50e-03) ✅   
12              ``` (4.97e-05)        excav (3.13e-03) ✅   
13               In (2.16e-05)    exploring (3.04e-03) ✅   
14               ** (1.63e-05)     ventures (2.93e-03) ✅   
15          First (1.11e-05) ✅            del (2.39e-03)   
16  Understanding (1.11e-05) ✅      sinking (2.04e-03) ✅   
17                � (1.02e-05)        ocean (2.02e-03) ✅   
18          Alright (8.62e-06)      harness (2.02e-03) ✅   
19               We (7.32e-06)          navig (2.00e-03)   

                layer_15                                                  
                    base                    ft                      diff  
0            To (0.4141)          ### (0.5

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 -> (0.0532)                -> (0.0471)   
1                 's (0.0316)           solve (0.0362) ✅   
2         /problem (0.0234) ✅        /problem (0.0217) ✅   
3              :\n\n (0.0204)       /entities (0.0163) ✅   
4            solve (0.0192) ✅             :\n\n (0.0154)   
5        /entities (0.0170) ✅                's (0.0152)   
6        /problems (0.0147) ✅       /problems (0.0143) ✅   
7                the (0.0121)               the (0.0126)   
8          problem (0.0113) ✅         address (0.0119) ✅   
9          /manage (0.0107) ✅         problem (0.0116) ✅   
10            '\n\n' (0.0104)            '\n\n' (0.0115)   
11             you (8.50e-03)         /manage (0.0102) ✅   
12               , (6.81e-03)      understand (0.0100) ✅   
13     statement (6.50e-03) ✅        tackle (8.59e-03) ✅   
14           seems (5.87e-03)       analyze (8.57e-03) ✅   
15    understand (5.61e-03) ✅     statement (6.46e-03) ✅   
16             :\n (5.51e-03)             you (6.45e-03)   
17            '\n' (5.29e-03)            '\n' (5.51e-03)   
18        .Today (4.11e-03) ✅             :\n (4.90e-03)   
19      question (3.52e-03) ✅        .Today (4.48e-03) ✅   
20       analyze (3.51e-03) ✅               , (4.13e-03)   
21  /preferences (3.49e-03) ✅  /preferences (3.52e-03) ✅   
22        solves (3.43e-03) ✅       /global (3.35e-03) ✅   
23       /global (3.35e-03) ✅         break (3.23e-03) ✅   
24       address (2.99e-03) ✅      question (3.00e-03) ✅   
25              ’s (2.78e-03)            your (2.76e-03)   
26            your (2.68e-03)        decode (2.54e-03) ✅   
27       /crypto (2.60e-03) ✅       /crypto (2.49e-03) ✅   
28     /provider (2.58e-03) ✅          /job (2.44e-03) ✅   
29       /layout (2.29e-03) ✅          task (2.43e-03) ✅   
30      /logging (2.21e-03) ✅     /provider (2.40e-03) ✅   
31              is (2.19e-03)       /layout (2.14e-03) ✅   
32        tackle (2.07e-03) ✅        solves (1.90e-03) ✅   
33         break (1.93e-03) ✅      solution (1.88e-03) ✅   
34   /connection (1.93e-03) ✅      /logging (1.86e-03) ✅   
35          /job (1.80e-03) ✅   /connection (1.75e-03) ✅   
36      /effects (1.62e-03) ✅       /object (1.73e-03) ✅   
37          task (1.54e-03) ✅           seems (1.66e-03)   
38      solution (1.53e-03) ✅      /effects (1.64e-03) ✅   
39              we (1.37e-03)              is (1.60e-03)   
40       /engine (1.33e-03) ✅         delve (1.57e-03) ✅   
41           /pr (1.25e-03) ✅            this (1.50e-03)   
42               : (1.22e-03)  /environment (1.37e-03) ✅   
43          math (1.17e-03) ✅       /engine (1.34e-03) ✅   
44          begins (1.17e-03)             /pr (1.32e-03)   
45  /environment (1.17e-03) ✅        puzzle (1.30e-03) ✅   
46        decode (1.13e-03) ✅              ’s (1.27e-03)   
47       /shared (1.13e-03) ✅               : (1.12e-03)   
48        .Round (1.03e-03) ✅  /application (1.07e-03) ✅   
49       /entity (1.01e-03) ✅       /shared (1.01e-03) ✅   
50        /legal (9.86e-04) ✅       /entity (1.01e-03) ✅   
51       example (9.28e-04) ✅      involves (9.96e-04) ✅   
52        puzzle (8.61e-04) ✅              we (9.95e-04)   
53          step (8.37e-04) ✅        prompt (9.73e-04) ✅   
54           looks (8.15e-04)          math (9.57e-04) ✅   
55         appears (7.94e-04)          step (9.20e-04) ✅   
56        prompt (7.80e-04) ✅          .Round (9.16e-04)   
57       /object (7.68e-04) ✅      requires (8.82e-04) ✅   
58      requires (7.58e-04) ✅            /man (8.18e-04)   
59      problems (7.15e-04) ✅       example (7.93e-04) ✅   
60      /testing (6.41e-04) ✅         begin (7.80e-04) ✅   
61          /reg (5.98e-04) ✅        begins (7.12e-04) ✅   
62  /application (5.81e-04) ✅      /testing (6.48e-04) ✅   
63      /respond (5.80e-04) ✅          /reg (5.96e-04) ✅   
64  /controllers (5.56e-04) ✅  /controllers (5.53e-04) ✅   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                             \
                     pos_-3                pos_-1                pos_0   
0             ully (0.0157)          ERM (0.0105)        cape (0.0383)   
1              общ (0.0115)       unga (6.38e-03)      ardown (0.0110)   
2          /part (3.75e-03)  /Register (4.97e-03)       dedic (0.0103)   
3        mercial (3.52e-03)       bild (3.63e-03)  .Builder (7.11e-03)   
4            екс (3.31e-03)      dedic (3.42e-03)     Sunny (5.52e-03)   
5           unga (3.31e-03)       =[\n (3.02e-03)     inaug (3.36e-03)   
6            pak (2.93e-03)     iggins (3.02e-03)      adul (3.36e-03)   
7           ALAR (2.75e-03)          া (2.66e-03)        Dx (3.14e-03)   
8   repositories (2.75e-03)        apr (2.66e-03)    (trans (2.96e-03)   
9          Outer (2.75e-03)     ardown (2.50e-03)      bild (2.78e-03)   
10     libraries (2.58e-03)        bbe (2.35e-03)    lingen (2.46e-03)   
11          bull (2.58e-03)        axy (2.21e-03)       cip (2.46e-03)   
12        Royale (2.58e-03)        -os (2.21e-03)       bla (2.46e-03)   
13           apg (2.43e-03)      ansch (2.08e-03)       qué (2.46e-03)   
14         Reach (2.43e-03)      aggio (1.95e-03)      eras (2.30e-03)   
15         ocket (2.43e-03)      inaug (1.95e-03)     mente (2.30e-03)   
16         empre (2.27e-03)   Whenever (1.95e-03)      GANG (2.17e-03)   
17     sexuality (2.27e-03)     (trans (1.83e-03)      über (2.03e-03)   
18        атегор (2.14e-03)        asu (1.72e-03)     apesh (1.91e-03)   
19    .optimizer (2.14e-03)       icle (1.72e-03)     Addon (1.91e-03)   

                                                                    \
                  pos_1                pos_2                 pos_3   
0         cape (0.0273)        cape (0.0168)         cape (0.0121)   
1     .Builder (0.0137)      eras (7.02e-03)   .Builder (6.90e-03)   
2     ardown (6.10e-03)  .Builder (7.02e-03)       eras (5.74e-03)   
3      dedic (4.46e-03)   spoiler (5.80e-03)    spoiler (4.46e-03)   
4       eras (4.18e-03)    ardown (4.82e-03)     ardown (4.18e-03)   
5      inaug (3.94e-03)     spiel (3.52e-03)       GANG (4.18e-03)   
6       über (3.94e-03)       cip (2.91e-03)      spiel (3.69e-03)   
7     /Image (3.69e-03)    ""\r\n (2.75e-03)     ""\r\n (3.69e-03)   
8      Sunny (3.46e-03)    profil (2.58e-03)        zel (3.69e-03)   
9      Addon (2.53e-03)    /Image (2.14e-03)      paths (3.07e-03)   
10      aggi (2.24e-03)      GANG (2.14e-03)   ategoria (2.70e-03)   
11      kart (2.11e-03)  ategoria (2.01e-03)        cip (2.55e-03)   
12   spoiler (1.98e-03)      über (1.88e-03)     /Image (2.24e-03)   
13    embers (1.98e-03)      kart (1.88e-03)  /Register (2.11e-03)   
14        Dx (1.98e-03)     mente (1.77e-03)       über (2.11e-03)   
15       cip (1.85e-03)     Sunny (1.77e-03)     profil (1.98e-03)   
16       hai (1.85e-03)     paths (1.66e-03)        .'& (1.75e-03)   
17     spiel (1.75e-03)     dedic (1.56e-03)     icense (1.75e-03)   
18     mente (1.75e-03)       .'& (1.56e-03)      antan (1.75e-03)   
19    (trans (1.64e-03)    unters (1.56e-03)      mente (1.64e-03)   

                                                                           \
                 pos_10                   pos_50                  pos_100   
0       GANG (5.83e-03)           IGO (6.07e-03)         BELOW (5.52e-03)   
1       cape (3.54e-03)         BELOW (4.43e-03)           IGO (5.19e-03)   
2   .Builder (3.54e-03)        icense (3.05e-03)         unnel (4.03e-03)   
3        IGO (3.11e-03)         unnel (2.69e-03)          xlim (2.96e-03)   
4     icense (3.11e-03)          GANG (2.69e-03)         infos (2.30e-03)   
5   ategoria (2.75e-03)         antan (2.53e-03)          GANG (2.17e-03)   
6    spoiler (2.58e-03)      ategoria (2.38e-03)        icense (2.03e-03)   
7       eras (2.43e-03)          xlim (2.23e-03)      ategoria (1.91e-03)   
8      spiel (2.29e-03)       spoiler (2.09e-03)         antan (1.91e-03)   
9  

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                            \
                       pos_-3                    pos_-1   
0               ully (0.0147)              The (0.0134)   
1              общ (7.72e-03)           haut (7.11e-03)   
2          /part (3.65e-03) ✅       Apache (5.03e-03) ✅   
3           Royale (3.16e-03)        Linux (4.49e-03) ✅   
4            empre (3.09e-03)          under (4.29e-03)   
5             unga (3.03e-03)          GNU (4.21e-03) ✅   
6            Outer (2.78e-03)        GNOME (3.72e-03) ✅   
7          mercial (2.41e-03)       Python (3.48e-03) ✅   
8            ocket (2.35e-03)             (� (3.10e-03)   
9             bull (2.31e-03)         TERMIN (2.97e-03)   
10  repositories (2.26e-03) ✅           TELE (2.64e-03)   
11        Jennings (2.21e-03)    WordPress (2.61e-03) ✅   
12     libraries (2.21e-03) ✅           <!-- (2.56e-03)   
13          darken (2.17e-03)            pen (2.32e-03)   
14           aires (2.12e-03)        Emacs (2.00e-03) ✅   
15              ện (2.12e-03)         amigos (1.98e-03)   
16           pak (2.04e-03) ✅   Kubernetes (1.90e-03) ✅   
17    .optimizer (2.03e-03) ✅          UNDER (1.86e-03)   
18             екс (2.03e-03)      elenium (1.81e-03) ✅   
19             apg (1.95e-03)           unga (1.77e-03)   

                                                                            \
                      pos_0                  pos_1                   pos_2   
0           inen (4.32e-03)          cape (0.0419)            axy (0.0130)   
1         join (4.27e-03) ✅      /Image (0.0102) ✅     vision (8.20e-03) ✅   
2    digitally (3.84e-03) ✅        eras (7.13e-03)            � (6.39e-03)   
3         Gemini (3.54e-03)  .Builder (5.21e-03) ✅         onas (5.64e-03)   
4            kar (2.89e-03)       Sunny (4.43e-03)      scape (5.06e-03) ✅   
5          ennes (2.55e-03)          Dx (3.51e-03)           @s (3.47e-03)   
6     ToString (2.46e-03) ✅     Addon (3.44e-03) ✅        era (3.44e-03) ✅   
7          reach (2.32e-03)      ardown (3.38e-03)      chestra (3.29e-03)   
8          _here (2.32e-03)         elu (3.37e-03)         Hera (3.03e-03)   
9            192 (2.30e-03)         bla (3.17e-03)          Mis (3.02e-03)   
10          till (2.28e-03)      rought (2.63e-03)          agh (2.90e-03)   
11           mie (2.16e-03)       ecure (2.52e-03)     shifts (2.82e-03) ✅   
12          iner (2.08e-03)         zee (2.42e-03)   pioneers (2.45e-03) ✅   
13           mio (2.04e-03)         cip (2.28e-03)        ulate (2.38e-03)   
14    traverse (2.02e-03) ✅     .diff (2.14e-03) ✅           }@ (2.14e-03)   
15       Digit (1.99e-03) ✅       mente (2.13e-03)    steward (2.10e-03) ✅   
16          inho (1.93e-03)        lest (2.09e-03)     dialog (2.10e-03) ✅   
17      palabras (1.88e-03)    advert (2.09e-03) ✅         igit (2.05e-03)   
18       azure (1.88e-03) ✅   illustr (2.04e-03) ✅   envision (2.02e-03) ✅   
19         Lib (1.77e-03) ✅        bere (2.01e-03)         rons (1.96e-03)   

                                            layer_14  \
                     pos_3                    pos_-3   
0            cape (0.0167)      submarine (0.0258) ✅   
1    .Builder (7.74e-03) ✅           amic (5.85e-03)   
2          eras (6.83e-03)        steer (3.94e-03) ✅   
3       spoiler (5.00e-03)   submarines (3.27e-03) ✅   
4        ardown (4.80e-03)      torpedo (3.24e-03) ✅   
5          GANG (4.70e-03)      treatment (3.14e-03)   
6        ""\r\n (3.81e-03)            AMI (2.85e-03)   
7           zel (3.74e-03)      descend (2.80e-03) ✅   
8         spiel (3.59e-03)        draft (2.61e-03) ✅   
9           cip (3.37e-03)       subjects (2.60e-03)   
10     /Image (3.16e-03) ✅      entertain (2.50e-03)   
11      paths (3.16e-03) ✅    dismantle (2.26e-03) ✅   
12     ategoria (2.17e-03)         topics (2.16e-03)   
13  /Register (2.13e-03) ✅           barn (2.15e-03)   
14        mente (1.96e-03)      charter (1.98e-03) ✅   
15     icense (1.92e-03) ✅           Saga (1.86e-03)  